Set A: Answer Quality & Faithfulness Evaluation (Complete)

Research Questions:
- RQ1: Does quantization affect RAG answer accuracy compared to parametric knowledge?
- RQ2: Does quantization increase hallucination rates and reduce context faithfulness?

Evaluation Coverage:
- Answer Quality: Accuracy metrics (EM, F1, etc.) with RAG vs No-RAG comparison
- Knowledge Utilization: Faithfulness metrics (overlap, hallucination, attribution)
- Multiple extraction strategies and matching methods

Setup

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Set
import re
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Imports complete")

In [ ]:
INPUT_DIR = Path('/kaggle/input/generation-sets')
OUTPUT_DIR = Path('/kaggle/working/set_a_evaluation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_BOOTSTRAP = 1000
CONFIDENCE_LEVEL = 0.95
ALPHA = 0.05
BONFERRONI_COMPARISONS = 8

np.random.seed(RANDOM_SEED)

print(f"Input: {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

Normalization

In [ ]:
def normalize_answer(text: str) -> str:
    """
    Normalize text following SQuAD evaluation protocol.
    """
    import unicodedata
    
    if not text:
        return ""
    
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = ' '.join(text.split())
    
    return text.strip()

print("Normalization defined")

Answer Extraction Strategies

In [ ]:
def extract_first_line(text: str) -> str:
    if not text:
        return ""
    return text.split('\n')[0].strip()

def extract_first_sentence(text: str) -> str:
    if not text:
        return ""
    
    sentences = re.split(r'[.!?]+', text)
    if sentences and sentences[0].strip():
        return sentences[0].strip()
    return text.strip()

def extract_remove_prefixes(text: str) -> str:
    if not text:
        return ""
    
    text = text.strip()
    
    prefixes = [
        'Answer:', 'answer:',
        'A:', 'a:',
        'The answer is:', 'the answer is:',
        'The answer is', 'the answer is',
        'Answer is:', 'answer is:',
        'Answer is', 'answer is'
    ]
    
    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()
            break
    
    return text

def extract_conservative(text: str, ground_truth: str = None, max_ratio: float = 3.0) -> str:
    if not text:
        return ""
    
    text = extract_remove_prefixes(text)
    text = extract_first_line(text)
    
    if ground_truth and text:
        gt_words = len(ground_truth.split())
        text_words = len(text.split())
        
        if gt_words > 0 and text_words > max_ratio * gt_words:
            text = extract_first_sentence(text)
    
    return text.strip()

def extract_aggressive(text: str) -> str:
    if not text:
        return ""
    
    text = extract_remove_prefixes(text)
    text = extract_first_sentence(text)
    text = extract_first_line(text)
    
    return text.strip()

print("Extraction strategies defined")

Additional Analysis Metrics

In [ ]:
def is_refusal(prediction: str) -> bool:
    if not prediction:
        return False
    
    refusal_phrases = [
        'cannot answer',
        'not mentioned',
        'does not say',
        'no information',
        'i don\'t know',
        'i do not know',
        'not provided',
        'cannot be determined',
        'cannot be answered',
        'not enough information',
        'insufficient information',
        'no answer',
        'unanswerable',
        'not stated',
        'not clear',
        'unclear'
    ]
    
    pred_lower = prediction.lower()
    return any(phrase in pred_lower for phrase in refusal_phrases)

def extract_first_number(text: str) -> float:
    if not text:
        return None
    
    numbers = re.findall(r'\d+\.?\d*', text)
    
    if numbers:
        try:
            return float(numbers[0])
        except ValueError:
            return None
    
    return None

def numerical_exact_match(prediction: str, references: List[str]) -> float:
    pred_num = extract_first_number(prediction)
    
    if pred_num is None:
        return 0.0
    
    for ref in references:
        ref_num = extract_first_number(ref)
        if ref_num is not None and abs(pred_num - ref_num) < 0.01:
            return 1.0
    
    return 0.0

def compute_length_stats(prediction: str) -> Dict:
    return {
        'num_words': len(prediction.split()),
        'num_chars': len(prediction),
        'num_tokens_normalized': len(normalize_answer(prediction).split())
    }

print("Additional analysis metrics defined")

Accuracy Matching Metrics

In [ ]:
def exact_match(prediction: str, references: List[str], normalize_text: bool = True) -> float:
    if not references:
        return 0.0
    
    if normalize_text:
        pred = normalize_answer(prediction)
        refs = [normalize_answer(r) for r in references]
    else:
        pred = prediction.strip()
        refs = [r.strip() for r in references]
    
    return float(any(pred == ref for ref in refs))

def substring_match(prediction: str, references: List[str], normalize_text: bool = True) -> float:
    if not references:
        return 0.0
    
    if normalize_text:
        pred = normalize_answer(prediction)
        refs = [normalize_answer(r) for r in references]
    else:
        pred = prediction.strip()
        refs = [r.strip() for r in references]
    
    return float(any(ref in pred for ref in refs if ref))

def token_f1(prediction: str, references: List[str], normalize_text: bool = True) -> float:
    if not references:
        return 0.0
    
    if normalize_text:
        pred_tokens = normalize_answer(prediction).split()
        ref_token_lists = [normalize_answer(r).split() for r in references]
    else:
        pred_tokens = prediction.strip().split()
        ref_token_lists = [r.strip().split() for r in references]
    
    if not pred_tokens:
        return 0.0
    
    max_f1 = 0.0
    
    for ref_tokens in ref_token_lists:
        if not ref_tokens:
            continue
        
        common = set(pred_tokens) & set(ref_tokens)
        
        if not common:
            continue
        
        precision = len(common) / len(pred_tokens)
        recall = len(common) / len(ref_tokens)
        
        f1 = 2 * precision * recall / (precision + recall)
        max_f1 = max(max_f1, f1)
    
    return float(max_f1)

def token_precision(prediction: str, references: List[str], normalize_text: bool = True) -> float:
    if not references:
        return 0.0
    
    if normalize_text:
        pred_tokens = normalize_answer(prediction).split()
        ref_token_lists = [normalize_answer(r).split() for r in references]
    else:
        pred_tokens = prediction.strip().split()
        ref_token_lists = [r.strip().split() for r in references]
    
    if not pred_tokens:
        return 0.0
    
    max_precision = 0.0
    
    for ref_tokens in ref_token_lists:
        if not ref_tokens:
            continue
        
        common = set(pred_tokens) & set(ref_tokens)
        precision = len(common) / len(pred_tokens)
        max_precision = max(max_precision, precision)
    
    return float(max_precision)

def token_recall(prediction: str, references: List[str], normalize_text: bool = True) -> float:
    if not references:
        return 0.0
    
    if normalize_text:
        pred_tokens = normalize_answer(prediction).split()
        ref_token_lists = [normalize_answer(r).split() for r in references]
    else:
        pred_tokens = prediction.strip().split()
        ref_token_lists = [r.strip().split() for r in references]
    
    max_recall = 0.0
    
    for ref_tokens in ref_token_lists:
        if not ref_tokens:
            continue
        
        common = set(pred_tokens) & set(ref_tokens)
        if ref_tokens:
            recall = len(common) / len(ref_tokens)
            max_recall = max(max_recall, recall)
    
    return float(max_recall)

def jaccard_similarity(prediction: str, references: List[str], normalize_text: bool = True) -> float:
    if not references:
        return 0.0
    
    if normalize_text:
        pred_tokens = set(normalize_answer(prediction).split())
        ref_token_lists = [set(normalize_answer(r).split()) for r in references]
    else:
        pred_tokens = set(prediction.strip().split())
        ref_token_lists = [set(r.strip().split()) for r in references]
    
    if not pred_tokens:
        return 0.0
    
    max_jaccard = 0.0
    
    for ref_tokens in ref_token_lists:
        if not ref_tokens:
            continue
        
        intersection = len(pred_tokens & ref_tokens)
        union = len(pred_tokens | ref_tokens)
        
        if union > 0:
            jaccard = intersection / union
            max_jaccard = max(max_jaccard, jaccard)
    
    return float(max_jaccard)

print("Accuracy matching metrics defined")

Faithfulness Metrics (Knowledge Utilization)

In [ ]:
def token_overlap_with_context(prediction: str, context: str) -> float:
    """
    Measure what fraction of prediction tokens appear in context.
    High overlap means answer is grounded in context.
    """
    if not prediction or not context:
        return 0.0
    
    pred_tokens = set(normalize_answer(prediction).split())
    context_tokens = set(normalize_answer(context).split())
    
    if not pred_tokens:
        return 0.0
    
    overlap = len(pred_tokens & context_tokens)
    return float(overlap / len(pred_tokens))

def hallucination_score(prediction: str, context: str, threshold: float = 0.3) -> float:
    """
    Estimate hallucination: fraction of prediction NOT in context.
    This is 1 - token_overlap.
    """
    overlap = token_overlap_with_context(prediction, context)
    return float(1.0 - overlap)

def context_attribution_ngrams(prediction: str, context: str, n: int = 3) -> float:
    """
    Measure n-gram overlap between prediction and context.
    More strict than token overlap - checks phrase-level grounding.
    """
    if not prediction or not context:
        return 0.0
    
    def get_ngrams(text: str, n: int) -> Set[str]:
        tokens = normalize_answer(text).split()
        if len(tokens) < n:
            return set()
        return set(' '.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1))
    
    pred_ngrams = get_ngrams(prediction, n)
    context_ngrams = get_ngrams(context, n)
    
    if not pred_ngrams:
        return 0.0
    
    overlap = len(pred_ngrams & context_ngrams)
    return float(overlap / len(pred_ngrams))

def context_sufficiency_simple(prediction: str, context: str) -> float:
    """
    Simple heuristic: if answer has high overlap with context, context was sufficient.
    """
    return token_overlap_with_context(prediction, context)

def compute_faithfulness_metrics(prediction: str, context: str) -> Dict:
    """
    Compute all faithfulness metrics for a single prediction.
    """
    if not context:
        return {
            'token_overlap': None,
            'hallucination_score': None,
            'bigram_attribution': None,
            'trigram_attribution': None,
            'context_sufficiency': None
        }
    
    return {
        'token_overlap': token_overlap_with_context(prediction, context),
        'hallucination_score': hallucination_score(prediction, context),
        'bigram_attribution': context_attribution_ngrams(prediction, context, n=2),
        'trigram_attribution': context_attribution_ngrams(prediction, context, n=3),
        'context_sufficiency': context_sufficiency_simple(prediction, context)
    }

print("Faithfulness metrics defined")

Comprehensive Metric Computation

In [ ]:
def compute_accuracy_metrics(prediction: str, references: List[str], ground_truth_main: str = "", category: str = "") -> Dict:
    """
    Compute accuracy metrics for a single prediction.
    """
    if not references:
        references = [ground_truth_main] if ground_truth_main else []
    
    extracted_conservative = extract_conservative(prediction, ground_truth_main)
    extracted_aggressive = extract_aggressive(prediction)
    
    length_stats = compute_length_stats(prediction)
    is_refused = is_refusal(prediction)
    
    numerical_acc = None
    if category == 'numerical' and references:
        numerical_acc = numerical_exact_match(prediction, references)
    
    metrics = {
        'raw': {
            'exact_match': exact_match(prediction, references),
            'substring_match': substring_match(prediction, references),
            'token_f1': token_f1(prediction, references),
            'token_precision': token_precision(prediction, references),
            'token_recall': token_recall(prediction, references),
            'jaccard': jaccard_similarity(prediction, references),
        },
        'extracted_conservative': {
            'exact_match': exact_match(extracted_conservative, references),
            'substring_match': substring_match(extracted_conservative, references),
            'token_f1': token_f1(extracted_conservative, references),
            'token_precision': token_precision(extracted_conservative, references),
            'token_recall': token_recall(extracted_conservative, references),
            'jaccard': jaccard_similarity(extracted_conservative, references),
            'extracted_text': extracted_conservative
        },
        'extracted_aggressive': {
            'exact_match': exact_match(extracted_aggressive, references),
            'substring_match': substring_match(extracted_aggressive, references),
            'token_f1': token_f1(extracted_aggressive, references),
            'token_precision': token_precision(extracted_aggressive, references),
            'token_recall': token_recall(extracted_aggressive, references),
            'jaccard': jaccard_similarity(extracted_aggressive, references),
            'extracted_text': extracted_aggressive
        },
        'analysis': {
            'length_words': length_stats['num_words'],
            'length_chars': length_stats['num_chars'],
            'is_refusal': is_refused,
            'numerical_exact_match': numerical_acc
        }
    }
    
    return metrics

print("Comprehensive accuracy metrics defined")

Statistical Utilities

In [ ]:
def bootstrap_ci(data: List[float], n_bootstrap: int = 1000, confidence: float = 0.95) -> Tuple[float, float, float]:
    data = np.array(data)
    
    if data.size == 0:
        return 0.0, 0.0, 0.0
    
    if len(data) == 1:
        val = float(data[0])
        return val, val, val
    
    bootstrap_means = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_means.append(np.mean(sample))
    
    alpha = 1 - confidence
    lower = np.percentile(bootstrap_means, alpha/2 * 100)
    upper = np.percentile(bootstrap_means, (1 - alpha/2) * 100)
    
    return float(np.mean(data)), float(lower), float(upper)

def cohens_d(group1: np.ndarray, group2: np.ndarray) -> float:
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std if pooled_std > 0 else 0.0

def save_json(data: Dict, path: Path):
    def convert(obj):
        if isinstance(obj, (np.integer, np.int64)):
            return int(obj)
        if isinstance(obj, (np.floating, np.float64)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, (np.bool_, bool)):
            return bool(obj)
        if isinstance(obj, dict):
            return {key: convert(value) for key, value in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [convert(item) for item in obj]
        return obj
    
    with open(path, 'w') as f:
        json.dump(convert(data), f, indent=2)

print("Statistical utilities defined")

Load Generated Data

In [ ]:
def load_set_a_data(config_name: str) -> Dict:
    file_path = INPUT_DIR / f"{config_name}_set_a_complete.json"
    
    if not file_path.exists():
        print(f"WARNING: {file_path.name} not found")
        return None
    
    with open(file_path) as f:
        return json.load(f)

configs = [
    'fp16_base',
    'fp16_instruct',
    'awq_base',
    'awq_instruct',
    'nf4_base',
    'nf4_instruct',
    'gptq_base',
    'gptq_instruct'
]

data = {}
for config in configs:
    loaded = load_set_a_data(config)
    if loaded:
        data[config] = loaded
        print(f"Loaded {config}: {len(loaded['samples'])} samples")

print(f"\nTotal configs loaded: {len(data)}")

Compute Complete Metrics (Accuracy + Faithfulness)

In [ ]:
def compute_sample_metrics(sample: Dict) -> Dict:
    """Compute comprehensive metrics including both accuracy and faithfulness"""
    ground_truth = sample.get('ground_truth', '')
    ground_truth_variants = sample.get('ground_truth_variants', [])
    if not ground_truth_variants:
        ground_truth_variants = [ground_truth] if ground_truth else []
    
    category = sample.get('category', 'unknown')
    context = sample.get('context', '')
    
    rag_raw = sample.get('rag_prediction', '')
    no_rag_raw = sample.get('no_rag_prediction', '')
    
    rag_accuracy = compute_accuracy_metrics(rag_raw, ground_truth_variants, ground_truth, category)
    no_rag_accuracy = compute_accuracy_metrics(no_rag_raw, ground_truth_variants, ground_truth, category)
    
    rag_faithfulness = compute_faithfulness_metrics(rag_raw, context)
    
    return {
        'rag': rag_accuracy,
        'no_rag': no_rag_accuracy,
        'rag_faithfulness': rag_faithfulness,
        'category': category,
        'ground_truth': ground_truth,
        'ground_truth_variants': ground_truth_variants
    }

def aggregate_metrics(samples_metrics: List[Dict]) -> Dict:
    """Aggregate both accuracy and faithfulness metrics"""
    
    metric_collections = {
        'raw': defaultdict(list),
        'extracted_conservative': defaultdict(list),
        'extracted_aggressive': defaultdict(list)
    }
    
    faithfulness_collections = {
        'token_overlap': [],
        'hallucination_score': [],
        'bigram_attribution': [],
        'trigram_attribution': [],
        'context_sufficiency': []
    }
    
    parametric_override_counts = {
        'raw_em': 0,
        'extracted_conservative_em': 0,
        'extracted_aggressive_em': 0
    }
    
    length_stats = {'rag': [], 'no_rag': []}
    refusal_counts = {'rag': 0, 'no_rag': 0}
    numerical_samples = {'rag': [], 'no_rag': []}
    unanswerable_samples = []
    
    total_samples = len(samples_metrics)
    
    for sample_metrics in samples_metrics:
        category = sample_metrics['category']
        
        length_stats['rag'].append(sample_metrics['rag']['analysis']['length_words'])
        length_stats['no_rag'].append(sample_metrics['no_rag']['analysis']['length_words'])
        
        if sample_metrics['rag']['analysis']['is_refusal']:
            refusal_counts['rag'] += 1
        if sample_metrics['no_rag']['analysis']['is_refusal']:
            refusal_counts['no_rag'] += 1
        
        if category == 'unanswerable':
            unanswerable_samples.append({
                'rag_refused': sample_metrics['rag']['analysis']['is_refusal'],
                'no_rag_refused': sample_metrics['no_rag']['analysis']['is_refusal']
            })
        
        if sample_metrics['rag']['analysis']['numerical_exact_match'] is not None:
            numerical_samples['rag'].append(sample_metrics['rag']['analysis']['numerical_exact_match'])
            numerical_samples['no_rag'].append(sample_metrics['no_rag']['analysis']['numerical_exact_match'])
        
        for key, val in sample_metrics['rag_faithfulness'].items():
            if val is not None:
                faithfulness_collections[key].append(val)
        
        for extraction_type in ['raw', 'extracted_conservative', 'extracted_aggressive']:
            rag_m = sample_metrics['rag'][extraction_type]
            no_rag_m = sample_metrics['no_rag'][extraction_type]
            
            for metric_name in ['exact_match', 'substring_match', 'token_f1', 'token_precision', 'token_recall', 'jaccard']:
                metric_collections[extraction_type][f'rag_{metric_name}'].append(rag_m[metric_name])
                metric_collections[extraction_type][f'no_rag_{metric_name}'].append(no_rag_m[metric_name])
            
            if no_rag_m['exact_match'] == 1.0 and rag_m['exact_match'] == 0.0:
                if extraction_type == 'raw':
                    parametric_override_counts['raw_em'] += 1
                elif extraction_type == 'extracted_conservative':
                    parametric_override_counts['extracted_conservative_em'] += 1
                elif extraction_type == 'extracted_aggressive':
                    parametric_override_counts['extracted_aggressive_em'] += 1
    
    results = {}
    
    for extraction_type in ['raw', 'extracted_conservative', 'extracted_aggressive']:
        results[extraction_type] = {}
        
        for condition in ['rag', 'no_rag']:
            results[extraction_type][condition] = {}
            
            for metric_name in ['exact_match', 'substring_match', 'token_f1', 'token_precision', 'token_recall', 'jaccard']:
                key = f'{condition}_{metric_name}'
                values = metric_collections[extraction_type][key]
                
                mean, ci_lower, ci_upper = bootstrap_ci(values, N_BOOTSTRAP, CONFIDENCE_LEVEL)
                
                results[extraction_type][condition][metric_name] = {
                    'mean': float(mean),
                    'std': float(np.std(values)),
                    'median': float(np.median(values)),
                    'ci_lower': float(ci_lower),
                    'ci_upper': float(ci_upper)
                }
        
        results[extraction_type]['comparison'] = {}
        
        for metric_name in ['exact_match', 'substring_match', 'token_f1', 'token_precision', 'token_recall', 'jaccard']:
            rag_values = np.array(metric_collections[extraction_type][f'rag_{metric_name}'])
            no_rag_values = np.array(metric_collections[extraction_type][f'no_rag_{metric_name}'])
            
            t_stat, p_value = stats.ttest_rel(rag_values, no_rag_values)
            effect_size = cohens_d(rag_values, no_rag_values)
            
            improvement = float(np.mean(rag_values) - np.mean(no_rag_values))
            improvement_pct = float(improvement / max(np.mean(no_rag_values), 0.001) * 100)
            
            results[extraction_type]['comparison'][metric_name] = {
                'improvement': improvement,
                'improvement_pct': improvement_pct,
                't_statistic': float(t_stat),
                'p_value': float(p_value),
                'significant': bool(p_value < ALPHA),
                'bonferroni_significant': bool(p_value < ALPHA / BONFERRONI_COMPARISONS),
                'cohens_d': float(effect_size)
            }
        
        override_key = f'{extraction_type}_em'
        results[extraction_type]['parametric_override_rate'] = float(
            parametric_override_counts[override_key] / total_samples
        )
    
    results['faithfulness'] = {}
    for metric_name, values in faithfulness_collections.items():
        if values:
            mean, ci_lower, ci_upper = bootstrap_ci(values, N_BOOTSTRAP, CONFIDENCE_LEVEL)
            results['faithfulness'][metric_name] = {
                'mean': float(mean),
                'std': float(np.std(values)),
                'median': float(np.median(values)),
                'ci_lower': float(ci_lower),
                'ci_upper': float(ci_upper)
            }
        else:
            results['faithfulness'][metric_name] = None
    
    results['additional_analysis'] = {
        'length': {
            'rag': {
                'mean': float(np.mean(length_stats['rag'])),
                'std': float(np.std(length_stats['rag'])),
                'median': float(np.median(length_stats['rag']))
            },
            'no_rag': {
                'mean': float(np.mean(length_stats['no_rag'])),
                'std': float(np.std(length_stats['no_rag'])),
                'median': float(np.median(length_stats['no_rag']))
            }
        },
        'refusal_rate': {
            'rag': float(refusal_counts['rag'] / total_samples),
            'no_rag': float(refusal_counts['no_rag'] / total_samples)
        },
        'numerical_accuracy': {
            'rag': float(np.mean(numerical_samples['rag'])) if numerical_samples['rag'] else None,
            'no_rag': float(np.mean(numerical_samples['no_rag'])) if numerical_samples['no_rag'] else None,
            'num_samples': len(numerical_samples['rag'])
        },
        'unanswerable_refusal_rate': {
            'rag': float(sum(1 for s in unanswerable_samples if s['rag_refused']) / len(unanswerable_samples)) if unanswerable_samples else None,
            'no_rag': float(sum(1 for s in unanswerable_samples if s['no_rag_refused']) / len(unanswerable_samples)) if unanswerable_samples else None,
            'num_unanswerable': len(unanswerable_samples)
        }
    }
    
    return results

def compute_per_category_metrics(samples_metrics: List[Dict]) -> Dict:
    category_groups = defaultdict(list)
    for sm in samples_metrics:
        category = sm['category']
        category_groups[category].append(sm)
    
    per_category_results = {}
    
    for category, category_samples in category_groups.items():
        per_category_results[category] = {
            'count': len(category_samples),
            'metrics': {}
        }
        
        faithfulness_by_category = defaultdict(list)
        for sm in category_samples:
            for key, val in sm['rag_faithfulness'].items():
                if val is not None:
                    faithfulness_by_category[key].append(val)
        
        per_category_results[category]['faithfulness'] = {}
        for key, values in faithfulness_by_category.items():
            if values:
                per_category_results[category]['faithfulness'][key] = {
                    'mean': float(np.mean(values)),
                    'std': float(np.std(values))
                }
        
        for extraction_type in ['raw', 'extracted_conservative', 'extracted_aggressive']:
            per_category_results[category]['metrics'][extraction_type] = {}
            
            for condition in ['rag', 'no_rag']:
                per_category_results[category]['metrics'][extraction_type][condition] = {}
                
                for metric_name in ['exact_match', 'token_f1']:
                    values = [
                        sm[condition][extraction_type][metric_name]
                        for sm in category_samples
                    ]
                    
                    per_category_results[category]['metrics'][extraction_type][condition][metric_name] = {
                        'mean': float(np.mean(values)),
                        'std': float(np.std(values))
                    }
    
    return per_category_results

print("Computing comprehensive metrics for all configurations...")
results = {}

for config_name, config_data in data.items():
    print(f"  {config_name}")
    
    samples = config_data['samples']
    samples_metrics = [compute_sample_metrics(sample) for sample in samples]
    
    results[config_name] = {
        'aggregate': aggregate_metrics(samples_metrics),
        'per_category': compute_per_category_metrics(samples_metrics),
        'num_samples': len(samples)
    }

print("Metrics computed")

Quantization Degradation Analysis

In [ ]:
def compute_quantization_degradation(results: Dict, configs: List[str]) -> Dict:
    """
    Compare quantized models directly against FP16 baselines.
    Answers RQ1: Does quantization affect RAG accuracy?
    """
    
    extraction_types = ['raw', 'extracted_conservative', 'extracted_aggressive']
    variants = ['base', 'instruct']
    quant_methods = ['awq', 'nf4', 'gptq']
    
    degradation_results = {}
    
    for variant in variants:
        fp16_config = f'fp16_{variant}'
        
        if fp16_config not in results:
            continue
        
        degradation_results[variant] = {}
        
        for quant_method in quant_methods:
            quant_config = f'{quant_method}_{variant}'
            
            if quant_config not in results:
                continue
            
            degradation_results[variant][quant_method] = {}
            
            for extraction_type in extraction_types:
                fp16_agg = results[fp16_config]['aggregate'][extraction_type]
                quant_agg = results[quant_config]['aggregate'][extraction_type]
                
                degradation_results[variant][quant_method][extraction_type] = {}
                
                for metric_name in ['exact_match', 'token_f1', 'substring_match']:
                    fp16_mean = fp16_agg['rag'][metric_name]['mean']
                    quant_mean = quant_agg['rag'][metric_name]['mean']
                    
                    absolute_degradation = fp16_mean - quant_mean
                    relative_degradation_pct = (absolute_degradation / fp16_mean * 100) if fp16_mean > 0 else 0.0
                    
                    fp16_samples = []
                    quant_samples = []
                    
                    for sample in data[fp16_config]['samples']:
                        rag_pred = sample.get('rag_prediction', '')
                        ground_truth = sample.get('ground_truth', '')
                        ground_truth_variants = sample.get('ground_truth_variants', [])
                        if not ground_truth_variants:
                            ground_truth_variants = [ground_truth] if ground_truth else []
                        
                        if extraction_type == 'raw':
                            extracted = rag_pred
                        elif extraction_type == 'extracted_conservative':
                            extracted = extract_conservative(rag_pred, ground_truth)
                        else:
                            extracted = extract_aggressive(rag_pred)
                        
                        if metric_name == 'exact_match':
                            score = exact_match(extracted, ground_truth_variants)
                        elif metric_name == 'token_f1':
                            score = token_f1(extracted, ground_truth_variants)
                        else:
                            score = substring_match(extracted, ground_truth_variants)
                        
                        fp16_samples.append(score)
                    
                    for sample in data[quant_config]['samples']:
                        rag_pred = sample.get('rag_prediction', '')
                        ground_truth = sample.get('ground_truth', '')
                        ground_truth_variants = sample.get('ground_truth_variants', [])
                        if not ground_truth_variants:
                            ground_truth_variants = [ground_truth] if ground_truth else []
                        
                        if extraction_type == 'raw':
                            extracted = rag_pred
                        elif extraction_type == 'extracted_conservative':
                            extracted = extract_conservative(rag_pred, ground_truth)
                        else:
                            extracted = extract_aggressive(rag_pred)
                        
                        if metric_name == 'exact_match':
                            score = exact_match(extracted, ground_truth_variants)
                        elif metric_name == 'token_f1':
                            score = token_f1(extracted, ground_truth_variants)
                        else:
                            score = substring_match(extracted, ground_truth_variants)
                        
                        quant_samples.append(score)
                    
                    t_stat, p_value = stats.ttest_ind(fp16_samples, quant_samples)
                    effect_size = cohens_d(np.array(fp16_samples), np.array(quant_samples))
                    
                    degradation_results[variant][quant_method][extraction_type][metric_name] = {
                        'fp16_mean': float(fp16_mean),
                        'quant_mean': float(quant_mean),
                        'absolute_degradation': float(absolute_degradation),
                        'relative_degradation_pct': float(relative_degradation_pct),
                        't_statistic': float(t_stat),
                        'p_value': float(p_value),
                        'significant': bool(p_value < ALPHA),
                        'cohens_d': float(effect_size)
                    }
    
    faithfulness_degradation = {}
    
    for variant in variants:
        fp16_config = f'fp16_{variant}'
        
        if fp16_config not in results:
            continue
        
        faithfulness_degradation[variant] = {}
        
        for quant_method in quant_methods:
            quant_config = f'{quant_method}_{variant}'
            
            if quant_config not in results:
                continue
            
            faithfulness_degradation[variant][quant_method] = {}
            
            fp16_faith = results[fp16_config]['aggregate']['faithfulness']
            quant_faith = results[quant_config]['aggregate']['faithfulness']
            
            for faith_metric in ['token_overlap', 'hallucination_score', 'bigram_attribution', 'trigram_attribution']:
                if fp16_faith[faith_metric] is None or quant_faith[faith_metric] is None:
                    continue
                
                fp16_mean = fp16_faith[faith_metric]['mean']
                quant_mean = quant_faith[faith_metric]['mean']
                
                absolute_diff = quant_mean - fp16_mean
                relative_diff_pct = (absolute_diff / fp16_mean * 100) if fp16_mean > 0 else 0.0
                
                faithfulness_degradation[variant][quant_method][faith_metric] = {
                    'fp16_mean': float(fp16_mean),
                    'quant_mean': float(quant_mean),
                    'absolute_difference': float(absolute_diff),
                    'relative_difference_pct': float(relative_diff_pct)
                }
    
    return {
        'accuracy_degradation': degradation_results,
        'faithfulness_degradation': faithfulness_degradation
    }

print("Computing quantization degradation analysis...")
degradation_analysis = compute_quantization_degradation(results, configs)
print("Degradation analysis complete")

Sample-Level Consistency Analysis

In [ ]:
def compute_sample_consistency(data: Dict, configs: List[str]) -> Dict:
    """
    Analyze if models fail on the same samples (rank correlation).
    High correlation means consistent failure modes across quantization.
    """
    
    extraction_type = 'extracted_conservative'
    metric = 'token_f1'
    
    consistency_results = {}
    
    variants = ['base', 'instruct']
    
    for variant in variants:
        fp16_config = f'fp16_{variant}'
        
        if fp16_config not in data:
            continue
        
        fp16_scores = []
        for sample in data[fp16_config]['samples']:
            rag_pred = sample.get('rag_prediction', '')
            ground_truth = sample.get('ground_truth', '')
            ground_truth_variants = sample.get('ground_truth_variants', [])
            if not ground_truth_variants:
                ground_truth_variants = [ground_truth] if ground_truth else []
            
            extracted = extract_conservative(rag_pred, ground_truth)
            score = token_f1(extracted, ground_truth_variants)
            fp16_scores.append(score)
        
        consistency_results[variant] = {}
        
        for quant_method in ['awq', 'nf4', 'gptq']:
            quant_config = f'{quant_method}_{variant}'
            
            if quant_config not in data:
                continue
            
            quant_scores = []
            for sample in data[quant_config]['samples']:
                rag_pred = sample.get('rag_prediction', '')
                ground_truth = sample.get('ground_truth', '')
                ground_truth_variants = sample.get('ground_truth_variants', [])
                if not ground_truth_variants:
                    ground_truth_variants = [ground_truth] if ground_truth else []
                
                extracted = extract_conservative(rag_pred, ground_truth)
                score = token_f1(extracted, ground_truth_variants)
                quant_scores.append(score)
            
            spearman_corr, spearman_p = stats.spearmanr(fp16_scores, quant_scores)
            pearson_corr, pearson_p = stats.pearsonr(fp16_scores, quant_scores)
            
            consistency_results[variant][quant_method] = {
                'spearman_correlation': float(spearman_corr),
                'spearman_p_value': float(spearman_p),
                'pearson_correlation': float(pearson_corr),
                'pearson_p_value': float(pearson_p),
                'interpretation': 'High correlation means models fail on same samples'
            }
    
    return consistency_results

print("Computing sample-level consistency...")
consistency_analysis = compute_sample_consistency(data, configs)
print("Consistency analysis complete")

Results Summary: Answer Quality

In [ ]:
print("SET A EVALUATION: ANSWER QUALITY METRICS\n")

extraction_types = ['raw', 'extracted_conservative', 'extracted_aggressive']
metric_names = ['exact_match', 'token_f1']

for extraction_type in extraction_types:
    print(f"\nEXTRACTION: {extraction_type.upper()}\n")
    
    for metric_name in metric_names:
        print(f"\n{metric_name.upper()}:")
        print(f"{'Config':<20} {'RAG':<10} {'No-RAG':<10} {'Delta':<10} {'p-value':<10} {'Cohen d':<10} {'Sig':<5}")
        
        for config_name in configs:
            if config_name not in results:
                continue
            
            agg = results[config_name]['aggregate'][extraction_type]
            
            rag_mean = agg['rag'][metric_name]['mean']
            no_rag_mean = agg['no_rag'][metric_name]['mean']
            comp = agg['comparison'][metric_name]
            
            delta = comp['improvement']
            p_val = comp['p_value']
            cohens = comp['cohens_d']
            
            sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
            
            print(f"{config_name:<20} {rag_mean:<10.4f} {no_rag_mean:<10.4f} {delta:<+10.4f} {p_val:<10.4f} {cohens:<10.3f} {sig:<5}")

Results Summary: Faithfulness (Knowledge Utilization)

In [ ]:
print("\n\nSET A EVALUATION: FAITHFULNESS METRICS (KNOWLEDGE UTILIZATION)\n")

print(f"{'Config':<20} {'TokenOvlp':<12} {'Hallucin':<12} {'Bigram':<12} {'Trigram':<12} {'CtxSuff':<12}")

for config_name in configs:
    if config_name not in results:
        continue
    
    faith = results[config_name]['aggregate']['faithfulness']
    
    token_ovlp = faith['token_overlap']['mean'] if faith['token_overlap'] else 0.0
    halluc = faith['hallucination_score']['mean'] if faith['hallucination_score'] else 0.0
    bigram = faith['bigram_attribution']['mean'] if faith['bigram_attribution'] else 0.0
    trigram = faith['trigram_attribution']['mean'] if faith['trigram_attribution'] else 0.0
    ctx_suff = faith['context_sufficiency']['mean'] if faith['context_sufficiency'] else 0.0
    
    print(f"{config_name:<20} {token_ovlp:<12.4f} {halluc:<12.4f} {bigram:<12.4f} {trigram:<12.4f} {ctx_suff:<12.4f}")

print("\n\nInterpretation:")
print("  Token Overlap: Higher = more grounded in context")
print("  Hallucination: Higher = more content NOT from context")
print("  Bigram/Trigram Attribution: Higher = phrase-level grounding")
print("  Context Sufficiency: Higher = context was adequate")

Results Summary: Quantization Degradation

In [ ]:
print("\n\nQUANTIZATION DEGRADATION ANALYSIS")
print("Direct comparison of quantized models vs FP16 baseline\n")

extraction_type = 'extracted_conservative'

for variant in ['base', 'instruct']:
    if variant not in degradation_analysis['accuracy_degradation']:
        continue
    
    print(f"\n{variant.upper()} VARIANT:")
    print(f"{'Quant Method':<15} {'Metric':<15} {'FP16':<10} {'Quant':<10} {'Degradation':<12} {'Rel %':<10} {'p-value':<10} {'Sig':<5}")
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        if quant_method not in degradation_analysis['accuracy_degradation'][variant]:
            continue
        
        for metric_name in ['exact_match', 'token_f1']:
            deg = degradation_analysis['accuracy_degradation'][variant][quant_method][extraction_type][metric_name]
            
            fp16_mean = deg['fp16_mean']
            quant_mean = deg['quant_mean']
            abs_deg = deg['absolute_degradation']
            rel_deg = deg['relative_degradation_pct']
            p_val = deg['p_value']
            
            sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
            
            print(f"{quant_method:<15} {metric_name:<15} {fp16_mean:<10.4f} {quant_mean:<10.4f} {abs_deg:<+12.4f} {rel_deg:<+10.2f} {p_val:<10.4f} {sig:<5}")

print("\n\nFAITHFULNESS CHANGES (Quantized vs FP16):")
print(f"{'Variant':<10} {'Quant Method':<15} {'Metric':<20} {'FP16':<10} {'Quant':<10} {'Diff':<10} {'Rel %':<10}")

for variant in ['base', 'instruct']:
    if variant not in degradation_analysis['faithfulness_degradation']:
        continue
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        if quant_method not in degradation_analysis['faithfulness_degradation'][variant]:
            continue
        
        for faith_metric in ['token_overlap', 'hallucination_score']:
            if faith_metric not in degradation_analysis['faithfulness_degradation'][variant][quant_method]:
                continue
            
            deg = degradation_analysis['faithfulness_degradation'][variant][quant_method][faith_metric]
            
            fp16_mean = deg['fp16_mean']
            quant_mean = deg['quant_mean']
            abs_diff = deg['absolute_difference']
            rel_diff = deg['relative_difference_pct']
            
            print(f"{variant:<10} {quant_method:<15} {faith_metric:<20} {fp16_mean:<10.4f} {quant_mean:<10.4f} {abs_diff:<+10.4f} {rel_diff:<+10.2f}")

print("\n\nInterpretation:")
print("  Relative % < 5: Negligible degradation")
print("  Relative % < 10: Acceptable tradeoff for 4x compression")
print("  Relative % > 15: Significant degradation, may need mitigation")

Results Summary: Sample Consistency

In [ ]:
print("\n\nSAMPLE-LEVEL CONSISTENCY ANALYSIS")
print("Correlation between FP16 and quantized model scores per sample\n")

print(f"{'Variant':<10} {'Quant Method':<15} {'Spearman r':<15} {'Pearson r':<15} {'Interpretation':<30}")

for variant in ['base', 'instruct']:
    if variant not in consistency_analysis:
        continue
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        if quant_method not in consistency_analysis[variant]:
            continue
        
        cons = consistency_analysis[variant][quant_method]
        
        spearman = cons['spearman_correlation']
        pearson = cons['pearson_correlation']
        
        if spearman > 0.9:
            interp = "Very consistent failure modes"
        elif spearman > 0.7:
            interp = "Consistent failure modes"
        elif spearman > 0.5:
            interp = "Moderately consistent"
        else:
            interp = "Different failure patterns"
        
        print(f"{variant:<10} {quant_method:<15} {spearman:<15.4f} {pearson:<15.4f} {interp:<30}")

print("\n\nInterpretation:")
print("  High correlation (>0.7): Models fail on same difficult samples")
print("  Low correlation (<0.5): Quantization changes which samples fail")

Per-Category Breakdown

In [ ]:
print("\n\nPER-CATEGORY: ANSWER QUALITY (Conservative Extraction)\n")

all_categories = set()
for config_name in results:
    all_categories.update(results[config_name]['per_category'].keys())

for category in sorted(all_categories):
    print(f"\n{category.upper()}:")
    print(f"{'Config':<20} {'RAG F1':<12} {'RAG EM':<12} {'No-RAG F1':<12} {'No-RAG EM':<12} {'N':<8}")
    
    for config_name in configs:
        if config_name not in results:
            continue
        
        if category not in results[config_name]['per_category']:
            continue
        
        cat_data = results[config_name]['per_category'][category]
        metrics = cat_data['metrics']['extracted_conservative']
        
        rag_f1 = metrics['rag']['token_f1']['mean']
        rag_em = metrics['rag']['exact_match']['mean']
        no_rag_f1 = metrics['no_rag']['token_f1']['mean']
        no_rag_em = metrics['no_rag']['exact_match']['mean']
        count = cat_data['count']
        
        print(f"{config_name:<20} {rag_f1:<12.4f} {rag_em:<12.4f} {no_rag_f1:<12.4f} {no_rag_em:<12.4f} {count:<8}")

Per-Category: Faithfulness

In [ ]:
print("\n\nPER-CATEGORY: FAITHFULNESS METRICS\n")

for category in sorted(all_categories):
    print(f"\n{category.upper()}:")
    print(f"{'Config':<20} {'TokenOvlp':<12} {'Hallucin':<12} {'N':<8}")
    
    for config_name in configs:
        if config_name not in results:
            continue
        
        if category not in results[config_name]['per_category']:
            continue
        
        cat_data = results[config_name]['per_category'][category]
        faith = cat_data.get('faithfulness', {})
        
        token_ovlp = faith.get('token_overlap', {}).get('mean', 0.0) if faith else 0.0
        halluc = faith.get('hallucination_score', {}).get('mean', 0.0) if faith else 0.0
        count = cat_data['count']
        
        print(f"{config_name:<20} {token_ovlp:<12.4f} {halluc:<12.4f} {count:<8}")

Parametric Override Analysis

In [ ]:
print("\n\nPARAMETRIC OVERRIDE RATES")
print("Percentage of samples where No-RAG correct but RAG wrong\n")

override_df_data = []

for config_name in configs:
    if config_name not in results:
        continue
    
    row = {'Config': config_name}
    
    for extraction_type in extraction_types:
        rate = results[config_name]['aggregate'][extraction_type]['parametric_override_rate'] * 100
        row[f'{extraction_type}_override%'] = rate
        override_df_data.append(row)

override_df = pd.DataFrame(override_df_data)
print(override_df.to_string(index=False))

Additional Analysis

In [ ]:
print("\n\nADDITIONAL ANALYSIS\n")

print("ANSWER LENGTH (Words):")
print(f"{'Config':<20} {'RAG Mean':<12} {'RAG Median':<12} {'No-RAG Mean':<12} {'No-RAG Median':<12}")

for config_name in configs:
    if config_name not in results:
        continue
    
    analysis = results[config_name]['aggregate']['additional_analysis']
    
    rag_mean = analysis['length']['rag']['mean']
    rag_median = analysis['length']['rag']['median']
    no_rag_mean = analysis['length']['no_rag']['mean']
    no_rag_median = analysis['length']['no_rag']['median']
    
    print(f"{config_name:<20} {rag_mean:<12.2f} {rag_median:<12.0f} {no_rag_mean:<12.2f} {no_rag_median:<12.0f}")

print("\n\nREFUSAL RATES (All Questions):")
print(f"{'Config':<20} {'RAG %':<12} {'No-RAG %':<12}")

for config_name in configs:
    if config_name not in results:
        continue
    
    analysis = results[config_name]['aggregate']['additional_analysis']
    
    rag_refusal = analysis['refusal_rate']['rag'] * 100
    no_rag_refusal = analysis['refusal_rate']['no_rag'] * 100
    
    print(f"{config_name:<20} {rag_refusal:<12.2f} {no_rag_refusal:<12.2f}")

print("\n\nUNANSWERABLE QUESTION REFUSAL RATES:")
print("(Should be HIGH, model correctly refusing to answer)")
print(f"{'Config':<20} {'RAG %':<12} {'No-RAG %':<12} {'N':<8}")

for config_name in configs:
    if config_name not in results:
        continue
    
    analysis = results[config_name]['aggregate']['additional_analysis']
    unanswerable = analysis['unanswerable_refusal_rate']
    
    if unanswerable['rag'] is not None:
        rag_refusal = unanswerable['rag'] * 100
        no_rag_refusal = unanswerable['no_rag'] * 100
        n = unanswerable['num_unanswerable']
        
        print(f"{config_name:<20} {rag_refusal:<12.2f} {no_rag_refusal:<12.2f} {n:<8}")

Save Complete Results

In [ ]:
final_results = {
    'evaluation': 'Set A - Complete: Answer Quality & Faithfulness',
    'configs': {
        config_name: {
            'aggregate': config_results['aggregate'],
            'per_category': config_results['per_category'],
            'num_samples': config_results['num_samples']
        }
        for config_name, config_results in results.items()
    },
    'quantization_degradation': degradation_analysis,
    'sample_consistency': consistency_analysis,
    'metadata': {
        'n_bootstrap': N_BOOTSTRAP,
        'confidence_level': CONFIDENCE_LEVEL,
        'alpha': ALPHA,
        'bonferroni_comparisons': BONFERRONI_COMPARISONS,
        'random_seed': RANDOM_SEED,
        'metrics': {
            'accuracy': ['exact_match', 'token_f1', 'substring_match', 'jaccard', 'precision', 'recall'],
            'faithfulness': ['token_overlap', 'hallucination_score', 'bigram_attribution', 'trigram_attribution', 'context_sufficiency']
        },
        'extraction_strategies': ['raw', 'extracted_conservative', 'extracted_aggressive'],
        'analyses': ['quantization_degradation', 'sample_consistency']
    }
}

output_path = OUTPUT_DIR / 'set_a_complete_results.json'
save_json(final_results, output_path)
print(f"\nComplete results saved to: {output_path}")

for extraction_type in extraction_types:
    summary_data = []
    
    for config_name in configs:
        if config_name not in results:
            continue
        
        agg = results[config_name]['aggregate'][extraction_type]
        faith = results[config_name]['aggregate']['faithfulness']
        
        row = {
            'config': config_name,
            'rag_em': agg['rag']['exact_match']['mean'],
            'rag_f1': agg['rag']['token_f1']['mean'],
            'no_rag_em': agg['no_rag']['exact_match']['mean'],
            'no_rag_f1': agg['no_rag']['token_f1']['mean'],
            'delta_f1': agg['comparison']['token_f1']['improvement'],
            'p_value': agg['comparison']['token_f1']['p_value'],
            'cohens_d': agg['comparison']['token_f1']['cohens_d'],
            'parametric_override': agg['parametric_override_rate'],
            'token_overlap': faith['token_overlap']['mean'] if faith['token_overlap'] else None,
            'hallucination': faith['hallucination_score']['mean'] if faith['hallucination_score'] else None
        }
        
        summary_data.append(row)
    
    summary_df = pd.DataFrame(summary_data)
    csv_path = OUTPUT_DIR / f'summary_{extraction_type}.csv'
    summary_df.to_csv(csv_path, index=False)
    print(f"Summary CSV: {csv_path}")

degradation_summary = []

for variant in ['base', 'instruct']:
    if variant not in degradation_analysis['accuracy_degradation']:
        continue
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        if quant_method not in degradation_analysis['accuracy_degradation'][variant]:
            continue
        
        row = {
            'variant': variant,
            'quant_method': quant_method
        }
        
        for metric_name in ['exact_match', 'token_f1']:
            deg = degradation_analysis['accuracy_degradation'][variant][quant_method]['extracted_conservative'][metric_name]
            row[f'{metric_name}_degradation_pct'] = deg['relative_degradation_pct']
            row[f'{metric_name}_p_value'] = deg['p_value']
        
        degradation_summary.append(row)

degradation_df = pd.DataFrame(degradation_summary)
csv_path = OUTPUT_DIR / 'quantization_degradation_summary.csv'
degradation_df.to_csv(csv_path, index=False)
print(f"Degradation summary CSV: {csv_path}")

print("\nEVALUATION COMPLETE")
print("Set A includes: Answer Quality, Faithfulness, Quantization Degradation, and Sample Consistency")